# Étape 2 — Apprentissage supervisé

**Instance** : les 11 variables explicatives du sujet + mois + année.
**Cible** : température à 2 m du **mois suivant**, discrétisée en **3 intervalles de même largeur**.
**Split** : chronologique.

Modèles : KNN, SVM, forêt aléatoire, AdaBoost, vote dur et vote souple.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 140)

In [ ]:
from src import config as C
from src import supervised as sp

clean = pd.read_parquet(C.FILE_CLEAN)
print("features :", sp.FEATURES)

## 1. Décalage de la cible

`t2m` est décalée d'un mois **à l'intérieur de chaque point de grille**. Une vérification sur les dates écarte les cas où les deux mois ne se suivent pas (trou dans la série).

In [ ]:
frame = sp.build_frame(clean)
frame.loc[:5, ["time", "latitude", "longitude", "t2m", "t2m_target"]]

## 2. Split chronologique puis discrétisation équi-largeur

Les bornes sont calculées **sur le train seul** puis appliquées au test : les
calculer sur l'ensemble complet ferait fuir dans le train une information sur
l'étendue des températures futures. Les bornes extrêmes sont ouvertes (±∞) pour
absorber les valeurs de test hors étendue.

In [ ]:
train, test, edges = sp.prepare(clean)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(train["t2m_target"], bins=60, ax=axes[0])
for b in edges[1:-1]:
    axes[0].axvline(b, color="crimson", ls="--")
axes[0].set_title("Cible continue et bornes équi-largeur")

pd.concat([train["classe"].value_counts(normalize=True).rename("train"),
           test["classe"].value_counts(normalize=True).rename("test")], axis=1) \
  .reindex(C.BIN_LABELS).plot.bar(ax=axes[1], title="Répartition des classes")
plt.tight_layout();

## 3. Entraînement et comparaison

Tous les modèles sont entraînés sur le **même** sous-échantillon du train : le SVM est quadratique en nombre de lignes, et l'échantillon commun garde la comparaison à effectif égal. Mettre `sample_size=None` pour utiliser tout le train (long).

In [ ]:
table, fitted, reports = sp.evaluate(train, test)
table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
table.set_index("modele")[["accuracy", "f1_macro", "kappa"]].plot.bar(ax=ax)
ax.set_title("Comparaison des modèles (jeu de test)")
ax.set_ylim(0, 1);

## 4. Analyse du meilleur modèle

In [ ]:
best = table.loc[0, "modele"]
print(f"=== {best} ===")
print(reports[best]["report"])

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
cm = pd.DataFrame(reports[best]["confusion"], index=C.BIN_LABELS, columns=C.BIN_LABELS)
sns.heatmap(cm / cm.sum(axis=1).values[:, None], annot=True, fmt=".2f", cmap="Blues", ax=ax)
ax.set(xlabel="prédit", ylabel="réel", title=f"Matrice de confusion normalisée — {best}");

In [ ]:
rf = fitted["random_forest"]
pd.Series(rf.feature_importances_, index=sp.FEATURES).sort_values().plot.barh(
    figsize=(7, 5), title="Importance des variables (forêt aléatoire)");

## 5. Erreur dans le temps et dans l'espace

In [ ]:
pred = fitted[best].predict(test[sp.FEATURES])
err = test.assign(juste=(pred == test["classe"]).astype(float))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
err.groupby("time")["juste"].mean().plot(ax=axes[0], marker="o",
                                         title="Taux de bonne classification par mois")
carte = err.groupby(["latitude", "longitude"])["juste"].mean().reset_index()
s = axes[1].scatter(carte["longitude"], carte["latitude"], c=carte["juste"],
                    cmap="viridis", s=18)
plt.colorbar(s, ax=axes[1])
axes[1].set_title("Taux de bonne classification par point")
plt.tight_layout();